In [1]:
import os
import time
from pyspark.sql import SparkSession

In [2]:
MINIO_AK = os.environ.get('MINIO_ACCESS_KEY')
MINIO_SK = os.environ.get('MINIO_SECRET_KEY')
#MASTER_URL = os.environ.get('SPARK_MASTER', 'local[*]')

In [ ]:
# # Em vez de fixar '1g', você dimensiona de acordo com o plano escolhido
# WORKSPACE_RAM = "1500m"  # Pede 1.5GB (Deixa um respiro para o OS)
# WORKSPACE_CORES = "2"    # Pede os 2 Cores do Worker

# spark = SparkSession.builder \
#     .appName("Job_Jupyter_ArenaLake") \
#     .master("spark://spark-master:7077") \
#     .config("spark.executor.cores", WORKSPACE_CORES) \
#     .config("spark.executor.memory", WORKSPACE_RAM) \
#     .config("spark.driver.memory", "2g") \
#     .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
#     .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
#     .config("spark.hadoop.fs.s3a.access.key", MINIO_AK) \
#     .config("spark.hadoop.fs.s3a.secret.key", MINIO_SK) \
#     .config("spark.hadoop.fs.s3a.path.style.access", "true") \
#     .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
#     .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
#     .getOrCreate()

In [3]:
import os
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

WORKSPACE_RAM = os.environ.get('WORKSPACE_RAM', '3g')
WORKSPACE_CORES = os.environ.get('WORKSPACE_CORES', '2')

print(f"🚀 Inicializando Spark unificado: {WORKSPACE_CORES} Cores | {WORKSPACE_RAM} RAM")

spark = SparkSession.builder \
    .appName("Job_Jupyter_ArenaLake") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.cores", WORKSPACE_CORES) \
    .config("spark.executor.memory", WORKSPACE_RAM) \
    .config("spark.driver.memory", "1g") \
    .config("spark.cores.max", WORKSPACE_CORES) \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_AK) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SK) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Conectado na máquina unificada com sucesso!")

🚀 Inicializando Spark unificado: 2 Cores | 3g RAM
:: loading settings :: url = jar:file:/usr/local/lib/python3.13/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/coder/.ivy2/cache
The jars for the packages stored in: /home/coder/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e40062f6-fe93-41c1-98cb-2e27e02552ff;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (104ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (3882ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.Fin

✅ Conectado na máquina unificada com sucesso!


In [ ]:
caminho_origem = "s3a://bronze/MOCK_DATA.csv"
print(f"📖 Lendo o arquivo CSV de origem: {caminho_origem}")

df = spark.read.csv(caminho_origem, header=True, inferSchema=True)
df.show(5)

In [ ]:
caminho_destino = "s3a://bronze/tabelas/MOCK_DATA_PARQUET"
print(f"💾 Convertendo e salvando como Tabela Parquet em: {caminho_destino}")

# Salva particionado (se já existir, ele sobrescreve)
df.write.mode("overwrite").parquet(caminho_destino)
print("✅ Tabela salva com sucesso!")

In [ ]:
print(f"🔄 Lendo a nova Tabela Parquet para confirmar...")
df_novo = spark.read.parquet(caminho_destino)

# Vamos fazer uma contagem rápida por Gênero para gerar um "Job" visível pro cluster calcular
print("📊 Agrupando por Gênero:")
df_novo.groupBy("gender").count().show()

In [ ]:
# Define o caminho da tabela
caminho_destino = "s3a://bronze/tabelas/MOCK_DATA_PARQUET"

# Recupera o FileSystem que o Spark está usando
sc = spark.sparkContext

path_obj = sc._gateway.jvm.org.apache.hadoop.fs.Path(caminho_destino)
fs = path_obj.getFileSystem(sc._jsc.hadoopConfiguration())

# Deleta a pasta e tudo dentro dela recursivamente
if fs.exists(path_obj):
    fs.delete(path_obj, True)
    print(f"🗑️ Tabela em {caminho_destino} deletada com sucesso!")
else:
    print("⚠️ O caminho especificado não existe.")


In [ ]:
spark.stop()